# Open-arXiv Full-Dataset FAISS Embedding Index

This Colab notebook builds a persistent vector index for `open-index/open-arxiv` using `google/embeddinggemma-300m`, FAISS, and Google Drive.

Persistent storage is written to:

`/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check/`

The FAISS index stores the embeddings. SQLite stores the source metadata and chunk text needed for retrieval and RAG.

## Runtime notes

- Use a GPU runtime for embedding generation.
- Run the default dry run first. It indexes `1,000` rows and validates checkpointing, retrieval, and metadata lookup.
- For the full run, set `RUN_MODE = "full"` in the config cell and rerun from the config cell onward.
- Full indexing processes about `2.99M` papers and can take many hours. Keep enough Google Drive space for the dataset cache, FAISS index, SQLite metadata DB, and checkpoints.
- `google/embeddinggemma-300m` may require a Hugging Face token. Set `HF_TOKEN` before loading the model if access fails.

In [ ]:
# Colab dependency setup. Re-run this cell after switching runtimes.
!pip -q install -U datasets sentence-transformers transformers accelerate faiss-cpu pyarrow tqdm openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN, optional unless the embedding model requires it: ').strip()
    if token:
        os.environ['HF_TOKEN'] = token

# Optional for the RAG answer-generation cells near the end.
# These support OpenAI-compatible endpoints, including a hosted API or your own vLLM server.
if not os.environ.get('LLM_API_KEY'):
    api_key = getpass.getpass('LLM_API_KEY for optional RAG generation, leave blank to skip: ').strip()
    if api_key:
        os.environ['LLM_API_KEY'] = api_key


In [ ]:
from pathlib import Path

DATASET_NAME = 'open-index/open-arxiv'
MODEL_ID = 'google/embeddinggemma-300m'
DIM = 768

CHUNK_CHARS = 1800
CHUNK_OVERLAP_CHARS = 250
DOCUMENT_PROMPT_TEMPLATE = 'title: {title} | text: '
QUERY_PROMPT = 'task: fact checking | query: '

# Dry run by default. Change to 'full' when you are ready to process the complete dataset.
RUN_MODE = 'dry_run'  # 'dry_run' or 'full'
MAX_ROWS = 1_000 if RUN_MODE == 'dry_run' else None

# Embedding and FAISS settings. Tune BATCH_SIZE down if Colab runs out of GPU memory.
BATCH_SIZE = 64
TRAINING_SAMPLE_CHUNKS = 50_000
NLIST = 4096
PQ_M = 96
PQ_BITS = 8
NPROBE = 32

# Durable checkpoints. Full-run checkpoints copy FAISS and SQLite artifacts to Drive.
CHECKPOINT_EVERY_CHUNKS = 100_000
STOP_AFTER_CHECKPOINTS = None  # set to 1 for a manual checkpoint/resume test

DRIVE_ROOT = Path('/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check')
DRIVE_FAISS_DIR = DRIVE_ROOT / 'faiss'
DRIVE_METADATA_DIR = DRIVE_ROOT / 'metadata'
DRIVE_CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
DRIVE_DATASET_CACHE_DIR = DRIVE_ROOT / 'hf_cache'
DRIVE_MANIFEST_PATH = DRIVE_ROOT / 'manifest.json'
DRIVE_FAISS_PATH = DRIVE_FAISS_DIR / 'open_arxiv_ivfpq.faiss'
DRIVE_SQLITE_PATH = DRIVE_METADATA_DIR / 'open_arxiv_chunks.sqlite'

LOCAL_ROOT = Path('/content/scholarrag_open_arxiv_work')
LOCAL_FAISS_PATH = LOCAL_ROOT / 'open_arxiv_ivfpq.faiss'
LOCAL_SQLITE_PATH = LOCAL_ROOT / 'open_arxiv_chunks.sqlite'

for path in [DRIVE_FAISS_DIR, DRIVE_METADATA_DIR, DRIVE_CHECKPOINT_DIR, DRIVE_DATASET_CACHE_DIR, LOCAL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print({'run_mode': RUN_MODE, 'max_rows': MAX_ROWS, 'drive_root': str(DRIVE_ROOT)})

In [ ]:
import json
import math
import shutil
import sqlite3
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Iterable
from uuid import NAMESPACE_URL, uuid5

import faiss
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def load_manifest() -> dict[str, Any]:
    if DRIVE_MANIFEST_PATH.exists():
        return json.loads(DRIVE_MANIFEST_PATH.read_text())
    return {}

def write_manifest(manifest: dict[str, Any]) -> None:
    manifest = dict(manifest)
    manifest['updated_at'] = utc_now()
    tmp_path = DRIVE_MANIFEST_PATH.with_suffix('.json.tmp')
    tmp_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    tmp_path.replace(DRIVE_MANIFEST_PATH)

manifest = load_manifest()
manifest

In [ ]:
@dataclass(frozen=True)
class Paper:
    paper_id: str
    title: str
    abstract: str
    categories: list[str]
    update_date: str | None
    authors: list[str]

@dataclass(frozen=True)
class Chunk:
    point_id: str
    paper_id: str
    chunk_id: int
    title: str
    text: str
    categories: list[str]
    update_date: str | None
    authors: list[str]

def clean_text(value: str | None) -> str:
    return ' '.join((value or '').split())

def parse_authors(authors_parsed: str | None) -> list[str]:
    if not authors_parsed:
        return []
    try:
        raw_authors = json.loads(authors_parsed)
    except json.JSONDecodeError:
        return []
    authors = []
    for author in raw_authors:
        if not isinstance(author, list) or len(author) < 2:
            continue
        last = str(author[0] or '').strip()
        first = str(author[1] or '').strip()
        name = ' '.join(part for part in [first, last] if part)
        if name:
            authors.append(name)
    return authors

def normalize_record(record: dict[str, Any]) -> Paper | None:
    paper_id = str(record.get('id') or '').strip()
    title = clean_text(record.get('title'))
    abstract = clean_text(record.get('abstract'))
    if not paper_id or not abstract:
        return None
    categories = [c.strip() for c in str(record.get('categories') or '').split() if c.strip()]
    update_date = str(record.get('update_date') or '').strip() or None
    authors = parse_authors(record.get('authors_parsed'))
    return Paper(paper_id=paper_id, title=title, abstract=abstract, categories=categories, update_date=update_date, authors=authors)

def split_text(text: str, *, chunk_chars: int, overlap_chars: int) -> list[str]:
    if chunk_chars <= 0:
        raise ValueError('chunk_chars must be positive')
    if overlap_chars < 0 or overlap_chars >= chunk_chars:
        raise ValueError('overlap_chars must be non-negative and smaller than chunk_chars')
    text = clean_text(text)
    if not text:
        return []
    if len(text) <= chunk_chars:
        return [text]
    chunks = []
    start = 0
    step = chunk_chars - overlap_chars
    while start < len(text):
        raw_end = min(start + chunk_chars, len(text))
        end = raw_end
        if raw_end < len(text):
            whitespace = text.rfind(' ', start + max(chunk_chars // 2, 1), raw_end)
            if whitespace > start:
                end = whitespace
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if raw_end >= len(text):
            break
        start = max(end - overlap_chars, start + step)
    return chunks

def stable_point_id(paper_id: str, chunk_id: int) -> str:
    return str(uuid5(NAMESPACE_URL, f'scholarrag:open-arxiv:{paper_id}:{chunk_id}'))

def chunk_paper(paper: Paper) -> list[Chunk]:
    full_text = f'{paper.title}\n\n{paper.abstract}' if paper.title else paper.abstract
    texts = split_text(full_text, chunk_chars=CHUNK_CHARS, overlap_chars=CHUNK_OVERLAP_CHARS)
    return [
        Chunk(
            point_id=stable_point_id(paper.paper_id, idx),
            paper_id=paper.paper_id,
            chunk_id=idx,
            title=paper.title,
            text=text,
            categories=paper.categories,
            update_date=paper.update_date,
            authors=paper.authors,
        )
        for idx, text in enumerate(texts)
    ]

def format_document_for_embedding(chunk: Chunk) -> str:
    title = clean_text(chunk.title) or 'none'
    return DOCUMENT_PROMPT_TEMPLATE.format(title=title) + chunk.text.strip()

def format_query_for_embedding(query: str) -> str:
    return QUERY_PROMPT + query.strip()

In [ ]:
def connect_sqlite(path: Path = LOCAL_SQLITE_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute('pragma journal_mode=delete')
    conn.execute('pragma synchronous=normal')
    conn.execute('pragma temp_store=memory')
    return conn

def init_sqlite(conn: sqlite3.Connection) -> None:
    conn.executescript(
        '''
        create table if not exists chunks (
            vector_id integer primary key,
            point_id text unique not null,
            paper_id text not null,
            chunk_id integer not null,
            title text not null,
            text text not null,
            categories_json text not null,
            update_date text,
            authors_json text not null,
            created_at text not null
        );
        create index if not exists idx_chunks_paper_id on chunks(paper_id);
        create index if not exists idx_chunks_update_date on chunks(update_date);
        create table if not exists indexing_state (
            key text primary key,
            value text not null
        );
        '''
    )
    conn.commit()

def insert_metadata(conn: sqlite3.Connection, vector_ids: np.ndarray, chunks: list[Chunk]) -> None:
    rows = [
        (
            int(vector_id),
            chunk.point_id,
            chunk.paper_id,
            chunk.chunk_id,
            chunk.title,
            chunk.text,
            json.dumps(chunk.categories),
            chunk.update_date,
            json.dumps(chunk.authors),
            utc_now(),
        )
        for vector_id, chunk in zip(vector_ids, chunks)
    ]
    with conn:
        conn.executemany(
            '''
            insert or replace into chunks (
                vector_id, point_id, paper_id, chunk_id, title, text,
                categories_json, update_date, authors_json, created_at
            ) values (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            rows,
        )

def fetch_chunks_by_vector_ids(conn: sqlite3.Connection, vector_ids: Iterable[int]) -> list[dict[str, Any]]:
    ids = [int(x) for x in vector_ids if int(x) >= 0]
    if not ids:
        return []
    placeholders = ','.join('?' for _ in ids)
    rows = conn.execute(f'select * from chunks where vector_id in ({placeholders})', ids).fetchall()
    by_id = {int(row['vector_id']): dict(row) for row in rows}
    ordered = []
    for vector_id in ids:
        row = by_id.get(vector_id)
        if row:
            row['categories'] = json.loads(row.pop('categories_json') or '[]')
            row['authors'] = json.loads(row.pop('authors_json') or '[]')
            ordered.append(row)
    return ordered

In [ ]:
print('Loading dataset. First full download/cache can take a while...')
dataset = load_dataset(DATASET_NAME, split='train', cache_dir=str(DRIVE_DATASET_CACHE_DIR))
row_limit = len(dataset) if MAX_ROWS is None else min(MAX_ROWS, len(dataset))
print(f'Dataset rows available: {len(dataset):,}; this run will scan: {row_limit:,}')

print('Loading embedding model...')
model_kwargs = {}
try:
    import torch
    if torch.cuda.is_available():
        model_kwargs['torch_dtype'] = torch.bfloat16
except Exception:
    model_kwargs = {}

embedder = SentenceTransformer(
    MODEL_ID,
    token=os.environ.get('HF_TOKEN') or None,
    truncate_dim=DIM,
    model_kwargs=model_kwargs,
)

def encode_chunks(chunks: list[Chunk]) -> np.ndarray:
    texts = [format_document_for_embedding(chunk) for chunk in chunks]
    vectors = embedder.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.ndim != 2 or vectors.shape[1] != DIM:
        raise ValueError(f'Embedding shape mismatch: expected (*, {DIM}), got {vectors.shape}')
    return vectors

def encode_query(query: str) -> np.ndarray:
    vector = embedder.encode(
        [format_query_for_embedding(query)],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(vector, dtype=np.float32)

In [ ]:
def choose_nlist(train_count: int) -> int:
    if train_count < 1_000:
        return max(8, min(64, train_count // 8 or 8))
    enough_training = max(16, train_count // 39)
    return int(min(NLIST, enough_training))

def build_faiss_index(train_vectors: np.ndarray) -> faiss.IndexIDMap2:
    if train_vectors.dtype != np.float32:
        train_vectors = train_vectors.astype(np.float32)
    effective_nlist = choose_nlist(len(train_vectors))
    quantizer = faiss.IndexFlatIP(DIM)
    base_index = faiss.IndexIVFPQ(quantizer, DIM, effective_nlist, PQ_M, PQ_BITS, faiss.METRIC_INNER_PRODUCT)
    print(f'Training IndexIVFPQ with nlist={effective_nlist}, m={PQ_M}, nbits={PQ_BITS}, train_vectors={len(train_vectors):,}')
    base_index.train(train_vectors)
    base_index.nprobe = min(NPROBE, effective_nlist)
    return faiss.IndexIDMap2(base_index)

def set_nprobe(index: faiss.Index, nprobe: int = NPROBE) -> None:
    base = index.index if isinstance(index, faiss.IndexIDMap2) else index
    if hasattr(base, 'nprobe'):
        base.nprobe = min(int(nprobe), int(getattr(base, 'nlist', nprobe)))

def load_faiss_index(path: Path = LOCAL_FAISS_PATH) -> faiss.IndexIDMap2:
    index = faiss.read_index(str(path))
    set_nprobe(index, NPROBE)
    return index

def add_vectors(index: faiss.IndexIDMap2, conn: sqlite3.Connection, chunks: list[Chunk], next_vector_id: int) -> int:
    if not chunks:
        return next_vector_id
    vectors = encode_chunks(chunks)
    vector_ids = np.arange(next_vector_id, next_vector_id + len(chunks), dtype=np.int64)
    index.add_with_ids(vectors, vector_ids)
    insert_metadata(conn, vector_ids, chunks)
    return next_vector_id + len(chunks)

In [ ]:
def restore_artifacts_from_drive() -> None:
    if DRIVE_FAISS_PATH.exists():
        shutil.copy2(DRIVE_FAISS_PATH, LOCAL_FAISS_PATH)
        print(f'Restored FAISS index from {DRIVE_FAISS_PATH}')
    if DRIVE_SQLITE_PATH.exists():
        shutil.copy2(DRIVE_SQLITE_PATH, LOCAL_SQLITE_PATH)
        print(f'Restored SQLite metadata from {DRIVE_SQLITE_PATH}')

def checkpoint_artifacts(
    *,
    index: faiss.IndexIDMap2,
    conn: sqlite3.Connection,
    next_row_index: int,
    next_vector_id: int,
    indexed_chunks: int,
    indexed_documents: int,
    checkpoint_number: int,
    status: str,
) -> None:
    faiss.write_index(index, str(LOCAL_FAISS_PATH))
    sqlite_backup_path = LOCAL_ROOT / 'open_arxiv_chunks.backup.sqlite'
    if sqlite_backup_path.exists():
        sqlite_backup_path.unlink()
    backup_conn = sqlite3.connect(sqlite_backup_path)
    conn.backup(backup_conn)
    backup_conn.close()

    shutil.copy2(LOCAL_FAISS_PATH, DRIVE_FAISS_PATH)
    shutil.copy2(sqlite_backup_path, DRIVE_SQLITE_PATH)

    checkpoint_tag = f'{checkpoint_number:05d}_{indexed_chunks:012d}_chunks'
    checkpoint_faiss = DRIVE_CHECKPOINT_DIR / f'{checkpoint_tag}.faiss'
    checkpoint_sqlite = DRIVE_CHECKPOINT_DIR / f'{checkpoint_tag}.sqlite'
    shutil.copy2(LOCAL_FAISS_PATH, checkpoint_faiss)
    shutil.copy2(sqlite_backup_path, checkpoint_sqlite)

    write_manifest(
        {
            'status': status,
            'dataset_name': DATASET_NAME,
            'model_id': MODEL_ID,
            'embedding_dimension': DIM,
            'document_prompt_template': DOCUMENT_PROMPT_TEMPLATE,
            'query_prompt': QUERY_PROMPT,
            'chunk_chars': CHUNK_CHARS,
            'chunk_overlap_chars': CHUNK_OVERLAP_CHARS,
            'faiss_index_type': 'IndexIDMap2(IndexIVFPQ) cosine-via-normalized-inner-product',
            'faiss_nlist_requested': NLIST,
            'faiss_nlist_actual': int(getattr(index.index if isinstance(index, faiss.IndexIDMap2) else index, 'nlist', 0)),
            'faiss_pq_m': PQ_M,
            'faiss_pq_bits': PQ_BITS,
            'nprobe': NPROBE,
            'run_mode': RUN_MODE,
            'max_rows': MAX_ROWS,
            'next_row_index': next_row_index,
            'next_vector_id': next_vector_id,
            'indexed_documents': indexed_documents,
            'indexed_chunks': indexed_chunks,
            'checkpoint_number': checkpoint_number,
            'faiss_path': str(DRIVE_FAISS_PATH),
            'sqlite_path': str(DRIVE_SQLITE_PATH),
            'last_checkpoint_faiss': str(checkpoint_faiss),
            'last_checkpoint_sqlite': str(checkpoint_sqlite),
        }
    )
    print(f'Checkpoint {checkpoint_number} saved: documents={indexed_documents:,}, chunks={indexed_chunks:,}, next_row={next_row_index:,}')

In [ ]:
def chunks_for_row(row_index: int) -> list[Chunk]:
    paper = normalize_record(dataset[int(row_index)])
    if paper is None:
        return []
    return chunk_paper(paper)

def collect_training_chunks(start_row_index: int, row_limit: int) -> tuple[list[Chunk], int, int]:
    chunks: list[Chunk] = []
    document_count = 0
    next_row_index = start_row_index
    progress = tqdm(range(start_row_index, row_limit), desc='Collecting training chunks')
    for row_index in progress:
        row_chunks = chunks_for_row(row_index)
        if row_chunks:
            document_count += 1
            chunks.extend(row_chunks)
        next_row_index = row_index + 1
        progress.set_postfix(chunks=len(chunks))
        if len(chunks) >= TRAINING_SAMPLE_CHUNKS:
            break
    if not chunks:
        raise RuntimeError('No chunks were collected for FAISS training')
    return chunks, next_row_index, document_count

def run_indexing() -> dict[str, Any]:
    restore_artifacts_from_drive()
    current_manifest = load_manifest()
    start_row_index = int(current_manifest.get('next_row_index', 0))
    next_vector_id = int(current_manifest.get('next_vector_id', 0))
    indexed_documents = int(current_manifest.get('indexed_documents', 0))
    indexed_chunks = int(current_manifest.get('indexed_chunks', 0))
    checkpoint_number = int(current_manifest.get('checkpoint_number', 0))

    if current_manifest and current_manifest.get('run_mode') != RUN_MODE:
        raise RuntimeError(
            f"Existing artifacts were created with run_mode={current_manifest.get('run_mode')!r}, "
            f"but the current config has RUN_MODE={RUN_MODE!r}. Move or delete the Drive artifact folder "
            'before starting a different run mode.'
        )

    conn = connect_sqlite()
    init_sqlite(conn)

    if LOCAL_FAISS_PATH.exists() and current_manifest:
        index = load_faiss_index(LOCAL_FAISS_PATH)
        print(f'Resuming from row={start_row_index:,}, next_vector_id={next_vector_id:,}, existing_vectors={index.ntotal:,}')
    else:
        start_row_index = 0
        next_vector_id = 0
        indexed_documents = 0
        indexed_chunks = 0
        checkpoint_number = 0
        training_chunks, start_row_index, training_documents = collect_training_chunks(0, row_limit)
        training_vectors = encode_chunks(training_chunks)
        index = build_faiss_index(training_vectors)
        vector_ids = np.arange(0, len(training_chunks), dtype=np.int64)
        index.add_with_ids(training_vectors, vector_ids)
        insert_metadata(conn, vector_ids, training_chunks)
        next_vector_id = len(training_chunks)
        indexed_chunks = len(training_chunks)
        indexed_documents = training_documents
        checkpoint_number += 1
        checkpoint_artifacts(
            index=index,
            conn=conn,
            next_row_index=start_row_index,
            next_vector_id=next_vector_id,
            indexed_chunks=indexed_chunks,
            indexed_documents=indexed_documents,
            checkpoint_number=checkpoint_number,
            status='running',
        )
        if STOP_AFTER_CHECKPOINTS == 1:
            conn.close()
            return load_manifest()

    pending_chunks: list[Chunk] = []
    documents_since_checkpoint = 0
    chunks_at_last_checkpoint = indexed_chunks
    checkpoints_this_call = 0
    start_time = time.time()

    progress = tqdm(range(start_row_index, row_limit), initial=start_row_index, total=row_limit, desc='Indexing OpenArXiv')
    for row_index in progress:
        row_chunks = chunks_for_row(row_index)
        if row_chunks:
            pending_chunks.extend(row_chunks)
            indexed_documents += 1
            documents_since_checkpoint += 1

        if len(pending_chunks) >= BATCH_SIZE:
            batch = pending_chunks[:BATCH_SIZE]
            pending_chunks = pending_chunks[BATCH_SIZE:]
            next_vector_id = add_vectors(index, conn, batch, next_vector_id)
            indexed_chunks += len(batch)

        progress.set_postfix(documents=indexed_documents, chunks=indexed_chunks)

        if indexed_chunks - chunks_at_last_checkpoint >= CHECKPOINT_EVERY_CHUNKS:
            checkpoint_number += 1
            checkpoint_artifacts(
                index=index,
                conn=conn,
                next_row_index=row_index + 1,
                next_vector_id=next_vector_id,
                indexed_chunks=indexed_chunks,
                indexed_documents=indexed_documents,
                checkpoint_number=checkpoint_number,
                status='running',
            )
            checkpoints_this_call += 1
            chunks_at_last_checkpoint = indexed_chunks
            documents_since_checkpoint = 0
            if STOP_AFTER_CHECKPOINTS is not None and checkpoints_this_call >= STOP_AFTER_CHECKPOINTS:
                conn.close()
                return load_manifest()

    while pending_chunks:
        batch = pending_chunks[:BATCH_SIZE]
        pending_chunks = pending_chunks[BATCH_SIZE:]
        next_vector_id = add_vectors(index, conn, batch, next_vector_id)
        indexed_chunks += len(batch)

    checkpoint_number += 1
    checkpoint_artifacts(
        index=index,
        conn=conn,
        next_row_index=row_limit,
        next_vector_id=next_vector_id,
        indexed_chunks=indexed_chunks,
        indexed_documents=indexed_documents,
        checkpoint_number=checkpoint_number,
        status='complete' if row_limit == len(dataset) else 'dry_run_complete',
    )
    conn.close()
    elapsed = time.time() - start_time
    print(f'Indexing call complete in {elapsed / 60:.1f} minutes')
    return load_manifest()

In [ ]:
# Run this cell to build or resume the FAISS index.
# Dry run defaults to 1,000 rows. For the full run, set RUN_MODE = 'full' in the config cell.
result_manifest = run_indexing()
result_manifest

## Full run switch

After the dry run passes validation, clear the dry-run artifacts or move them aside, set `RUN_MODE = "full"` in the config cell, and rerun from the config cell through the indexing cell.

If you want to test checkpoint/resume before the full run, set `STOP_AFTER_CHECKPOINTS = 1`, run the indexing cell, restart the runtime, then set `STOP_AFTER_CHECKPOINTS = None` and rerun.

In [ ]:
def validate_manifest() -> None:
    m = load_manifest()
    required = {
        'dataset_name': DATASET_NAME,
        'model_id': MODEL_ID,
        'embedding_dimension': DIM,
        'document_prompt_template': DOCUMENT_PROMPT_TEMPLATE,
        'query_prompt': QUERY_PROMPT,
    }
    for key, expected in required.items():
        actual = m.get(key)
        assert actual == expected, f'{key}: expected {expected!r}, got {actual!r}'
    assert 'IndexIVFPQ' in m.get('faiss_index_type', ''), m.get('faiss_index_type')
    assert int(m.get('indexed_chunks', 0)) > 0
    print('Manifest validation passed')

def validate_artifacts() -> None:
    assert DRIVE_FAISS_PATH.exists(), DRIVE_FAISS_PATH
    assert DRIVE_SQLITE_PATH.exists(), DRIVE_SQLITE_PATH
    index = faiss.read_index(str(DRIVE_FAISS_PATH))
    conn = sqlite3.connect(DRIVE_SQLITE_PATH)
    row = conn.execute('select count(*) from chunks').fetchone()[0]
    conn.close()
    assert index.ntotal == row, f'FAISS vectors {index.ntotal:,} != SQLite rows {row:,}'
    print(f'Artifact validation passed: {index.ntotal:,} vectors and metadata rows')

validate_manifest()
validate_artifacts()

In [ ]:
# Load durable artifacts for retrieval. This cell can run in a fresh Colab session after mounting Drive.
retrieval_index = faiss.read_index(str(DRIVE_FAISS_PATH))
set_nprobe(retrieval_index, NPROBE)
retrieval_conn = connect_sqlite(DRIVE_SQLITE_PATH)
print(f'Loaded retrieval index with {retrieval_index.ntotal:,} vectors')

In [ ]:
def search_open_arxiv(question: str, *, top_k: int = 10, nprobe: int = NPROBE) -> list[dict[str, Any]]:
    set_nprobe(retrieval_index, nprobe)
    query_vector = encode_query(question)
    scores, ids = retrieval_index.search(query_vector, top_k)
    vector_ids = [int(x) for x in ids[0].tolist() if int(x) >= 0]
    rows = fetch_chunks_by_vector_ids(retrieval_conn, vector_ids)
    row_by_id = {int(row['vector_id']): row for row in rows}
    results = []
    for rank, (score, vector_id) in enumerate(zip(scores[0].tolist(), ids[0].tolist()), start=1):
        vector_id = int(vector_id)
        if vector_id < 0:
            continue
        row = row_by_id.get(vector_id)
        if row is None:
            raise RuntimeError(f'Missing SQLite metadata for vector_id={vector_id}')
        results.append({**row, 'rank': rank, 'score': float(score)})
    return results

def print_results(results: list[dict[str, Any]]) -> None:
    for result in results:
        categories = ' '.join(result.get('categories') or [])
        preview = clean_text(result['text'])[:500]
        print(f"[{result['rank']}] score={result['score']:.4f} paper_id={result['paper_id']} chunk={result['chunk_id']}")
        print(f"title: {result['title']}")
        print(f"categories: {categories} updated={result.get('update_date')}")
        print(f"preview: {preview}\n")

question = 'What evidence exists that retrieval augmented generation improves scientific question answering?'
results = search_open_arxiv(question, top_k=10)
print_results(results)

In [ ]:
smoke_questions = [
    'Does retrieval augmented generation improve scientific question answering?',
    'What papers discuss transformer models for protein structure prediction?',
    'Is there evidence that contrastive learning improves image representations?',
    'What methods are used for neural machine translation in low resource languages?',
    'How are graph neural networks used in molecular property prediction?',
]

for smoke_question in smoke_questions:
    smoke_results = search_open_arxiv(smoke_question, top_k=5)
    assert len(smoke_results) > 0
    assert all('paper_id' in row and row['text'] for row in smoke_results)
    print(f'PASS: {smoke_question} -> {len(smoke_results)} results')

In [ ]:
def build_rag_prompt(question: str, sources: list[dict[str, Any]]) -> str:
    context_blocks = []
    for source in sources:
        citation = f"[{source['rank']}] {source['paper_id']} - {source['title']}"
        context_blocks.append(f"{citation}\n{source['text']}")
    context = '\n\n'.join(context_blocks)
    return f"""You answer scientific questions using only the provided source abstracts.
If the sources do not contain enough evidence, say that the evidence is insufficient.
Cite sources with bracketed rank numbers like [1] or [2].

Question: {question}

Sources:
{context}

Grounded answer:"""

def generate_grounded_answer(question: str, *, top_k: int = 8) -> dict[str, Any]:
    sources = search_open_arxiv(question, top_k=top_k)
    prompt = build_rag_prompt(question, sources)

    api_key = os.environ.get('LLM_API_KEY') or os.environ.get('OPENAI_API_KEY')
    base_url = os.environ.get('LLM_BASE_URL') or os.environ.get('OPENAI_BASE_URL')
    model_name = os.environ.get('LLM_MODEL', 'gpt-4o-mini')
    if not api_key:
        return {
            'answer': 'LLM_API_KEY is not set. The retrieval context and prompt are returned for inspection.',
            'prompt': prompt,
            'sources': sources,
        }

    from openai import OpenAI
    client_kwargs = {'api_key': api_key}
    if base_url:
        client_kwargs['base_url'] = base_url
    client = OpenAI(**client_kwargs)
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {'role': 'system', 'content': 'You write concise grounded scientific answers with source citations.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0.2,
    )
    return {'answer': response.choices[0].message.content, 'prompt': prompt, 'sources': sources}

rag_question = 'What evidence exists that retrieval augmented generation improves scientific question answering?'
rag_result = generate_grounded_answer(rag_question, top_k=8)
print(rag_result['answer'])

## Where embeddings are stored

Embeddings are not saved as raw `.npy` shards. They are stored inside the FAISS index at:

`/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check/faiss/open_arxiv_ivfpq.faiss`

The matching source text and metadata are stored in SQLite at:

`/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check/metadata/open_arxiv_chunks.sqlite`

The manifest at `manifest.json` records the embedding model, prompts, FAISS parameters, row/chunk counts, and checkpoint state. Checkpoints live under `checkpoints/` so a long Colab job can resume from the last durable copy.